# HypeClipper v10 - Zero-Drift HD Shorts Engine
3-stage `0.000s`-locked subtitle sync, anti-hang timeouts, live progress logs, strict HD (no 360p), subtitle ON/OFF toggle, 5-sound dynamic SFX, clickbait peak thumbnails, and 10+ hour VOD support.
1. **Runtime > Change runtime type > T4 GPU**.
2. Run **Step 4.5** first (~8 seconds) to verify facecam framing + captions before running the full batch.

In [ ]:
#@title 1. Check GPU & Mount Google Drive
import shutil, subprocess, os
from google.colab import drive

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', gpu.stdout.strip())
else:
    print('No GPU detected; running on CPU.')

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/HypeClipper', exist_ok=True)

In [ ]:
#@title 2. Install HypeClipper v10 Engine, Fonts & Dependencies
import os, subprocess

subprocess.run(['apt-get', '-qq', 'update'], check=False)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'fonts-dejavu-core', 'fonts-liberation', 'fontconfig'], check=False)
subprocess.run(['fc-cache', '-f'], check=False)

!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
os.environ['PATH'] += ':/root/.deno/bin:/usr/local/bin'
!rm -rf hype-clipper && git clone https://github.com/Aromalajith8/hype-clipper.git hype-clipper
!pip install -q -U yt-dlp faster-whisper
print('HypeClipper v10 ready!')

In [ ]:
#@title 3. Stream Source (10+ Hour VOD Supported) & Free AI Setup
source = '' #@param {type:"string"}
fetch_mode = 'auto' #@param ['auto', 'surgical', 'full']
cookies_path = '/content/drive/MyDrive/cookies.txt' #@param {type:"string"}
ai_provider = 'auto' #@param ['auto', 'groq', 'gemini', 'ollama', 'heuristic']
ollama_model = 'qwen3.5:9b' #@param ['qwen3.5:9b', 'qwen3:8b', 'qwen2.5:7b']
groq_api_key = '' #@param {type:"string"}
gemini_api_key = '' #@param {type:"string"}

In [ ]:
#@title 4. Configure Layout, Facecam, Subtitles (ON/OFF), Dynamic SFX & Clarity
from IPython.display import HTML, display
layout = 'cam-top' #@param ['cam-top', 'cam-bottom', 'pip-top-right', 'pip-top-left', 'crop-focus', 'blur-bg']
cam_pos = 'mid-left' #@param ['mid-left', 'mid-right', 'top-left', 'top-right', 'bottom-left', 'bottom-right', 'custom']
cam_w = 0.26 #@param {type:"slider", min:0.12, max:0.50, step:0.02}
cam_h = 0.38 #@param {type:"slider", min:0.12, max:0.60, step:0.02}
cam_x = 0.00 #@param {type:"slider", min:0.00, max:0.80, step:0.02}
cam_y = 0.38 #@param {type:"slider", min:0.00, max:0.80, step:0.02}
enable_subtitles = True #@param {type:"boolean"}
add_sfx = True #@param {type:"boolean"}
min_height = 1080 #@param [1080, 1440, 2160] {type:"raw"}
min_len = 25 #@param {type:"slider", min:20, max:40, step:5}
max_len = 45 #@param {type:"slider", min:25, max:50, step:5}
max_clips = 10 #@param {type:"slider", min:1, max:80, step:1}
caption_style = 'beast-bold' #@param ['auto', 'bold-pop', 'beast-bold', 'karaoke-highlight', 'word-by-word', 'two-line-stack', 'highlight-box', 'neon-glow', 'impact-shake', 'emoji-pop', 'typewriter', 'bounce', 'gradient-stroke', 'comic-burst', 'minimal-clean', 'gaming-glitch', 'none']
hook_style = 'mid-card' #@param ['mid-card', 'bold-box', 'outline-glow', 'top-bar', 'none']
display(HTML('\n<div style="background:#0d0f14;padding:14px;border-radius:10px;border:1px solid #262b38;font-family:sans-serif;color:#eee;">\n  <div style="font-weight:700;margin-bottom:10px;font-size:15px;">Caption & Hook Style Visual Preview (select in Step 4)</div>\n  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;text-align:center;">\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-weight:900;font-size:19px;color:#fff;">NO <span style="color:#ff0;">WAY</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">bold-pop</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:21px;color:#fff;">LET\'S <span style="color:#3f3;">GO</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">beast-bold</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-weight:800;font-size:15px;color:#fff;">HE IS <span style="color:#ff0;">CHEATING</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">karaoke-highlight</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:23px;color:#ff0;">INSANE</div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">word-by-word</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-weight:800;font-size:14px;line-height:1.2;color:#fff;">I CAN\'T<br>BELIEVE <span style="color:#ff0;">THIS</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">two-line-stack</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-weight:900;font-size:16px;"><span style="background:#ff0;color:#000;padding:2px 6px;">CLIP THAT</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">highlight-box</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-weight:900;font-size:17px;color:#fff;text-shadow:0 0 8px #0ff;">CHAT <span style="color:#0ff;">LOOK</span></div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">neon-glow</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="font-family:Impact,sans-serif;font-size:20px;color:#f44;">BOOM</div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">comic-burst</div></div>\n    <div style="background:#050608;padding:12px;border-radius:6px;"><div style="background:#fff;color:#000;font-weight:900;font-size:12px;padding:4px 8px;border-radius:6px;display:inline-block;">Speed starts STEALING</div><div style="font-size:11px;color:#8a93a6;margin-top:4px;">mid-card hook (split seam)</div></div>\n  </div>\n</div>\n'))
print(f'Config -> clips={int(max_clips)}, layout={layout}, subs={enable_subtitles} ({caption_style}), hook={hook_style}, sfx={add_sfx}, min_h={int(min_height)}p')

In [ ]:
#@title 4.5. 8-Second Facecam & Subtitle Calibration Preview
import glob, os, subprocess
from IPython.display import Image, display

preview_at_sec = 300 #@param {type:"number"}
assert source.strip(), 'Paste a stream URL or Drive video path in Step 3 first!'

cmd_prev = [
    'python3', '-u', 'hype-clipper/clipper.py', source.strip(),
    '--preview-layout', '--preview-sec', str(preview_at_sec),
    '--min-height', str(max(720, int(min_height))),
    '--layout', layout, '--cam-pos', cam_pos,
    '--cam-w', str(cam_w), '--cam-h', str(cam_h),
    '--cam-x', str(cam_x), '--cam-y', str(cam_y),
    '--caption-style', caption_style, '--hook-style', hook_style,
    '--out', '/content/drive/MyDrive/HypeClipper',
]
if not enable_subtitles:
    cmd_prev.append('--no-subtitles')
if cookies_path.strip() and os.path.exists(cookies_path.strip()):
    cmd_prev += ['--cookies', cookies_path.strip()]

proc = subprocess.Popen(cmd_prev, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

latest = sorted(glob.glob('/content/drive/MyDrive/HypeClipper/*'), key=os.path.getmtime)
if latest:
    p1 = os.path.join(latest[-1], 'preview_source_cambox.jpg')
    p2 = os.path.join(latest[-1], 'preview_rendered_short.jpg')
    if os.path.exists(p1):
        print('1) Green Box = Your Facecam Crop Area:')
        display(Image(p1, width=560))
    if os.path.exists(p2):
        print('2) Actual Rendered 1080x1920 Short Frame:')
        display(Image(p2, width=300))

In [ ]:
#@title 5. Run HypeClipper v10 Generator (Live Streaming Logs)
import glob, os, shutil, subprocess, time

assert source.strip(), 'Paste a stream URL or Drive video path in Step 3 first!'

if groq_api_key.strip():
    os.environ['GROQ_API_KEY'] = groq_api_key.strip()
if gemini_api_key.strip():
    os.environ['GEMINI_API_KEY'] = gemini_api_key.strip()

if ai_provider == 'ollama':
    if not shutil.which('ollama'):
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'zstd', 'curl'], check=False)
        subprocess.run('curl -fsSL -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz && mkdir -p /usr/local && tar -xzf /tmp/ollama.tgz -C /usr/local && rm -f /tmp/ollama.tgz', shell=True, check=False)
    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
    if os.path.exists(ollama_bin):
        subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        subprocess.run([ollama_bin, 'pull', ollama_model], check=False)

cmd = [
    'python3', '-u', 'hype-clipper/clipper.py', source.strip(),
    '--fetch-mode', fetch_mode,
    '--clips', str(int(max_clips)),
    '--min-len', str(int(min_len)),
    '--max-len', str(min(int(max_len), 50)),
    '--min-height', str(max(720, int(min_height))),
    '--layout', layout,
    '--cam-pos', cam_pos,
    '--cam-w', str(cam_w),
    '--cam-h', str(cam_h),
    '--cam-x', str(cam_x),
    '--cam-y', str(cam_y),
    '--caption-style', caption_style,
    '--hook-style', hook_style,
    '--out', '/content/drive/MyDrive/HypeClipper',
    '--ai-provider', ai_provider,
    '--ollama-model', ollama_model,
]

if not enable_subtitles:
    cmd.append('--no-subtitles')
if add_sfx:
    cmd.append('--sfx')
if cookies_path.strip() and os.path.exists(cookies_path.strip()):
    cmd += ['--cookies', cookies_path.strip()]

# Live real-time log streaming so you always see exact progress
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

latest_sessions = sorted(glob.glob('/content/drive/MyDrive/HypeClipper/*'), key=os.path.getmtime)
LAST_SESSION_DIR = latest_sessions[-1] if latest_sessions else None
print('Latest HypeClipper session:', LAST_SESSION_DIR)

In [ ]:
#@title 6. Preview Top Viral Clips & Clickbait Thumbnails
import glob, os
from IPython.display import Image, Video, display

sess = globals().get('LAST_SESSION_DIR')
if not sess or not os.path.isdir(sess):
    sessions = sorted(glob.glob('/content/drive/MyDrive/HypeClipper/*'), key=os.path.getmtime)
    sess = sessions[-1] if sessions else None

if sess:
    print('Top 10 Viral Folder:', os.path.join(sess, 'top_10_viral'))
    thumbs = sorted(glob.glob(os.path.join(sess, 'top_10_viral', '*_thumb.jpg')))[:3]
    for t in thumbs:
        print('Clickbait Thumbnail:', os.path.basename(t))
        display(Image(t, width=240))
    mp4s = sorted(glob.glob(os.path.join(sess, 'top_10_viral', '*.mp4')))[:3]
    for m in mp4s:
        print('Top Viral Preview:', os.path.basename(m))
        display(Video(m, embed=True, width=300))
else:
    print('No completed session found yet. Run Step 5 first.')